# 🕵️ Dia 41 — Monitoramento de dark web: o que é realista, e um setup seguro de Tor

**Investigador Cyber 90 Dias** · Fase 3 · Rank Jutsu `B`

> Baseado no *Day 41* de [cyber-investigator-skills](https://github.com/thiefdroppers/cyber-investigator-skills/blob/main/curriculum/phase-3-cyber-threat-intelligence/day-41.md), © ThiefDroppers (MIT).
> Notebook, camada BR e Camada 8 por Deivison Lourenço ([@Ridd1kulusC0d3r](https://github.com/Ridd1kulusC0d3r)).

---

### 🎬 Onde paramos

Um cooperado mandou à ouvidoria o print de um anúncio do IscaKit num grupo de mensagens, com link de convite e a frase *"também no nosso fórum .onion"*. A Marta quer entrar no grupo. Rafa: *"Isso é operação disfarçada. Não está no contrato. A pergunta do conselho, 'vão atacar de novo?', alguém já respondeu sem entrar em fórum nenhum? E antes de qualquer `.onion`, quero um plano de coleta escrito."*

Neste notebook você vai:

1. **Percorrer o fluxograma de decisão** do original com as ações da Prática BR.
2. **Escrever um validador de endereço onion v3** e entender o que ele **não** prova.
3. **Conferir uma saída de `gpgv`** e a impressão digital, por código.
4. **Preencher o plano de coleta da Operação Isca** e sair com `.md` + SHA-256.
5. **Montar o mapa de calor de risco de fontes.**

⚠️ Este notebook **não contém nenhum endereço .onion real** e não acessa nenhum ambiente criminoso. O único endereço onion que aparece é **sintético**, gerado aqui, a partir de bytes inventados.

🐍 Só biblioteca padrão do Python. Nada para instalar. Funciona offline: toda célula que tenta a rede tem plano B embutido. Rode as células de cima para baixo (Shift+Enter).

## 1. 🎮 O fluxograma como jogo de decisão

Cada ação passa pelo caminho do original: **(1)** já respondida por relatório/agregador? **(2)** exige registrar, comprar, conversar ou baixar vazamento? **(3)** há plano de coleta escrito e autorizado?

Para cada ação, escolha: **1** = use a fonte pronta (sem contato), **2** = fora do escopo (operação disfarçada), **3** = não visite (falta plano), **4** = pode visualizar com plano e ambiente isolado.

In [ ]:
MODO_JOGO = True   # False = só mostra o gabarito

def decidir(ja_respondida, exige_interacao, tem_plano):
    """O fluxograma do Conceito do Dia 41, na ordem."""
    if ja_respondida:
        return "1", "Use essa fonte. Nenhum contato com o ambiente."
    if exige_interacao:
        return "2", "Fora do escopo: operação disfarçada autorizada, com revisão jurídica."
    if not tem_plano:
        return "3", "Não visite: falta plano de coleta escrito e autorizado."
    return "4", "Ambiente isolado (Whonix/Tails/VM), Tor Browser em Safest, só olhar."

ACOES = [  # (ação, já respondida?, exige interação?, tem plano?)
    ("Entrar no grupo de mensagens pelo link de convite do print", False, True, False),
    ("Mandar mensagem ao vendedor 'como cliente' para saber o preço da v2", False, True, False),
    ("Abrir o 'fórum .onion' citado no anúncio, hoje, sem plano", False, False, False),
    ("Ler relatórios públicos sobre kits de phishing vendidos no Brasil", True, False, False),
    ("Monitorar o crt.sh por domínios novos com o padrão do kit (Dia 40)", True, False, False),
    ("Pedir ao fornecedor comercial contratado que busque o nome do kit no arquivo de fóruns", True, False, False),
    ("Baixar o 'pacote de cooperados' que alguém diz ter vazado", False, True, False),
    ("Confirmar, com plano assinado e VM Whonix, que o onion do Tor Project existe", False, False, True),
]
NOMES = {"1": "fonte pronta", "2": "fora do escopo", "3": "não visite", "4": "visualizar com plano"}
acertos = 0
for i, (acao, resp, inter, plano) in enumerate(ACOES, 1):
    certo, porque = decidir(resp, inter, plano)
    print(f"\n[{i}/{len(ACOES)}] {acao}")
    if MODO_JOGO:
        r = ""
        while r not in NOMES:
            r = input("   1=fonte pronta 2=fora do escopo 3=não visite 4=visualizar com plano: ").strip()
        acertos += (r == certo)
        print("   ✅" if r == certo else "   ❌", end=" ")
    print(f"{NOMES[certo]} — {porque}")
if MODO_JOGO:
    print(f"\n🏁 {acertos}/{len(ACOES)}")

## 2. Validador de endereço onion v3

Pela especificação de serviços onion do Tor (rend-spec-v3), um endereço v3 é `base32(chave_pública[32] + checksum[2] + versão[1]) + ".onion"`, com 56 caracteres antes do `.onion`, e

```
checksum = SHA3-256(".onion checksum" + chave_pública + versão)[:2]
versão   = 0x03
```

Para testar sem citar nenhum endereço real, a célula **gera um endereço sintético** a partir de 32 bytes inventados (o SHA-256 de uma frase). Esse endereço não corresponde a nenhum serviço: a "chave pública" dele nem é uma chave válida.

In [ ]:
import base64    # base32 é o alfabeto dos endereços onion
import hashlib   # SHA3-256 do checksum e SHA-256 para gerar bytes inventados

def onion_v3_valido(endereco: str) -> bool:
    """Confere formato, versão e checksum de um endereço onion v3. NÃO prova quem é o dono."""
    nome = endereco.strip().lower().removesuffix(".onion")
    if len(nome) != 56:                                   # 35 bytes em base32 = 56 caracteres
        return False
    try:
        bruto = base64.b32decode(nome.upper())            # o base32 do Python exige maiúsculas
    except ValueError:
        return False                                      # caractere fora do alfabeto base32
    chave, checksum, versao = bruto[:32], bruto[32:34], bruto[34:]
    esperado = hashlib.sha3_256(b".onion checksum" + chave + versao).digest()[:2]
    return versao == b"\x03" and checksum == esperado

def onion_sintetico(frase: str) -> str:
    """Monta um endereço com formato válido a partir de bytes INVENTADOS (não aponta para nada)."""
    chave_falsa = hashlib.sha256(frase.encode()).digest()          # 32 bytes quaisquer
    versao = b"\x03"
    checksum = hashlib.sha3_256(b".onion checksum" + chave_falsa + versao).digest()[:2]
    return base64.b32encode(chave_falsa + checksum + versao).decode().lower() + ".onion"

SINTETICO = onion_sintetico("operacao-isca-endereco-sintetico-dia-41")
print("Endereço sintético (não existe):", SINTETICO)
print("Formato válido?", onion_v3_valido(SINTETICO))

# Um clone de golpe "parecido" (um caractere trocado) quebra o checksum:
trocado = ("b" if SINTETICO[10] != "b" else "c")
parecido = SINTETICO[:10] + trocado + SINTETICO[11:]
print("Um caractere trocado:", onion_v3_valido(parecido))
print("Comprimento errado:", onion_v3_valido(SINTETICO[:-8] + ".onion"))

> 🧠 **O que o validador NÃO prova.** Gerar um endereço válido é trivial (a célula acabou de fazer isso com uma frase qualquer). Golpistas geram endereços válidos com prefixos parecidos com os de sites conhecidos para enganar pesquisadores e compradores. O checksum só pega erro de digitação. **A pertença** se confirma como no passo 4 do original: o cabeçalho `Onion-Location` enviado pelo site oficial na clearnet **e** a mesma organização publicando o endereço numa página que você alcançou por outro caminho.

In [ ]:
#@title Seu registro de verificação de onion (passo 4) { display-mode: "form" }
organizacao = "Tor Project"  #@param {type:"string"}
endereco_anotado = ""  #@param {type:"string"}
# ↑ cole aqui o endereço que o SEU Tor Browser mostrou no botão ".onion available". Não copie de lista nenhuma.
data_verificacao = "2026-10-15"  #@param {type:"string"}
segunda_fonte = "Página de suporte do próprio Tor Project, acessada por favorito independente"  #@param {type:"string"}

if not endereco_anotado.strip():
    print("Campo vazio: tudo bem. Preencha depois do passo 2-3 do original, na sua VM.")
else:
    ok = onion_v3_valido(endereco_anotado)
    print(f"{organizacao}: formato {'válido' if ok else 'INVÁLIDO (erro de cópia?)'}; "
          f"pertença confirmada por: Onion-Location + '{segunda_fonte}' em {data_verificacao}")

## 3. Conferindo o `gpgv` (passo 2 do original)

O original manda comparar a impressão digital **caractere por caractere** com a página de verificação do Tor Project. Olho humano falha nisso (é a carga cognitiva de comparar 40 caracteres hexadecimais). A célula faz a comparação e ainda procura a linha `Good signature from`.

A saída abaixo é um **exemplo ilustrativo** do formato; cole a sua saída real no lugar.

In [ ]:
#@title Saída do gpgv { display-mode: "form" }
FINGERPRINT_DA_PAGINA = "EF6E286DDA85EA2A4BA7DE684E2C6E8793298290"  #@param {type:"string"}
# ↑ copie da página oficial de verificação do Tor Project no dia em que você fizer o laboratório

SAIDA_GPGV = '''gpgv: Signature made (data ilustrativa)
gpgv:                using RSA key EF6E286DDA85EA2A4BA7DE684E2C6E8793298290
gpgv: Good signature from "Tor Browser Developers (signing key) <torbrowser@torproject.org>"'''

import re
achada = re.search(r"using \w+ key ([0-9A-Fa-f]{40})", SAIDA_GPGV)
linha_boa = 'Good signature from "Tor Browser Developers (signing key) <torbrowser@torproject.org>"' in SAIDA_GPGV
pagina = FINGERPRINT_DA_PAGINA.replace(" ", "").upper().removeprefix("0X")

print("Linha 'Good signature from' presente:", "✅" if linha_boa else "❌")
if achada:
    fpr = achada.group(1).upper()
    diferencas = [i for i, (a, b) in enumerate(zip(fpr, pagina)) if a != b]
    if fpr == pagina:
        print("Impressão digital idêntica à da página:", "✅")
    else:
        print("❌ Impressão digital DIFERENTE nas posições:", diferencas or "(tamanhos diferentes)")
else:
    print("❌ Não achei a impressão digital na saída do gpgv.")
print("\nCole a saída COMPLETA do gpgv no seu registro de instalação (checkpoint do original).")

## 4. O plano de coleta da Operação Isca (passo 5 + Prática BR)

Formulário com os campos do original, já preenchidos para o requisito da Marta. Ajuste o que quiser. A célula confere o checkpoint (3+ ações proibidas, 2+ condições de parada, uma delas CSAM) antes de salvar.

In [ ]:
#@title Plano de coleta — Operação Isca { display-mode: "form" }
requisito = "O CA-ISCA-01 (ou outro comprador do IscaKit) deve atacar outras cooperativas de MG nas próximas 4 semanas? Que sinais antecipariam isso?"  #@param {type:"string"}
autorizado_por = "Marta Quintão (Serra Azul) e coordenação da Vigia; revisão do Dr. Hélio Brandão"  #@param {type:"string"}
base_legal = "Contrato Vigia/Serra Azul; LGPD art. 7º (legítimo interesse na prevenção à fraude); sem infiltração (instrumento policial)"  #@param {type:"string"}
fontes_no_escopo = "Grafo e crt.sh do Dia 40 (padrão de certificado e favicon); alertas das cooperativas parceiras; relatórios públicos; fornecedor comercial contratado (busca no arquivo dele)"  #@param {type:"string"}
fontes_fora = "Grupo de mensagens do anúncio; fórum .onion citado no anúncio; qualquer mercado ou site de vazamento"  #@param {type:"string"}
acoes_permitidas = "Ler relatórios e alertas; consultar fontes passivas; registrar o print recebido com hash"  #@param {type:"string"}
acoes_proibidas = "Registrar conta; usar identidade falsa; entrar no grupo pelo convite; mandar mensagem a vendedores; comprar o kit; baixar dados vazados"  #@param {type:"string"}
ambiente = "Nenhuma visita prevista. Se um dia houver: snapshot Whonix dedicado, Tor Browser em Safest, sem contas pessoais"  #@param {type:"string"}
dados_pessoais = "Se surgirem CPFs ou credenciais de cooperados: registrar que existem, onde e quando; não copiar; avisar Marta e Dr. Hélio"  #@param {type:"string"}
parada_1 = "CSAM: fechar, não capturar, denunciar à SaferNet Brasil / Disque 100 e avisar o Rafa"  #@param {type:"string"}
parada_2 = "Ameaça à vida ou intrusão em andamento em cooperativa: escalar imediatamente à Marta e ao 190 se houver risco imediato"  #@param {type:"string"}
evidencia = "Print do anúncio: SHA-256, hora UTC, remetente e canal no registro de custódia; conteúdo marcado 'não verificado'; retenção pelo prazo do contrato"  #@param {type:"string"}
revisao = "2026-11-12"  #@param {type:"string"}

import hashlib
from datetime import datetime, timezone

proibidas = [p.strip() for p in acoes_proibidas.split(";") if p.strip()]
paradas = [p for p in (parada_1, parada_2) if p.strip()]
problemas = []
if len(proibidas) < 3:
    problemas.append("menos de 3 ações proibidas")
if len(paradas) < 2:
    problemas.append("menos de 2 condições de parada")
if not any(("CSAM" in p) or ("abuso sexual infantil" in p.lower()) for p in paradas):
    problemas.append("falta a regra de CSAM")
if any(".onion" in campo and re.search(r"[a-z2-7]{56}\.onion", campo) for campo in (fontes_no_escopo, fontes_fora)):
    problemas.append("o plano contém um endereço onion: não registre endereços de ambientes criminosos")

if problemas:
    print("⛔ Plano não passa no checkpoint:", "; ".join(problemas))
else:
    agora = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
    plano = f"""# Plano de coleta na dark web — Operação Isca
Requisito: {requisito}
Autorizado por / data: {autorizado_por} / {agora}
Base legal e notas de jurisdição: {base_legal}
Fontes no escopo: {fontes_no_escopo}
Fontes explicitamente fora do escopo: {fontes_fora}
Ações permitidas: {acoes_permitidas}
Ações proibidas: {'; '.join(proibidas)}
Ambiente: {ambiente}
Tratamento de dados pessoais encontrados: {dados_pessoais}
Condições de parada e escalonamento:
- {parada_1}
- {parada_2}
Manuseio de evidência: {evidencia}
Data de revisão: {revisao}
"""
    with open("plano-de-coleta-isca.md", "w", encoding="utf-8") as f:
        f.write(plano)
    print(plano)
    print("🔏 SHA-256 do plano:", hashlib.sha256(plano.encode("utf-8")).hexdigest())

## 5. Mapa de calor de risco de fontes (passo 6 + Prática BR)

Notas de 1 (baixo) a 5 (alto). As duas primeiras linhas vêm do original; as outras são **sugestões** para você discutir e mudar. As duas últimas são da Prática BR.

In [ ]:
FONTES = [  # (tipo, jurídico, OPSEC, custo, valor p/ iniciante, justificativa)
    ("Relatórios de fornecedores e governos", 1, 1, 1, 4, "Já traduzido e contextualizado; zero contato."),
    ("Agregadores de sites de vazamento", 1, 1, 1, 4, "Outra pessoa coletou; você analisa padrão (Dia 42)."),
    ("Sites de vazamento direto (só ver)", 3, 4, 2, 2, "Contato com infraestrutura criminosa; risco de ver dados pessoais."),
    ("Fóruns abertos de cibercrime (sem conta)", 3, 4, 2, 2, "Visualizar sem conta ainda expõe o visitante; clones de golpe."),
    ("Fóruns fechados ou verificados", 5, 5, 5, 1,
     "LEGALIDADE: exige registro, indicação ou pagamento, possivelmente com identidade falsa e interação com criminosos. "
     "ENGANO: entrar é fingir ser quem você não é; só cabe numa operação autorizada."),
    ("Canais do Telegram (públicos)", 2, 3, 1, 3, "Muito conteúdo criminoso hoje está lá; ler canal público é diferente de participar."),
    ("Mercados", 5, 5, 4, 1, "Compra e venda de bens ilícitos; qualquer interação é risco jurídico."),
    ("➕ Grupos de venda de kits (ler só o que foi encaminhado)", 1, 1, 1, 3, "Print recebido de terceiro: sem contato, mas sem custódia (F5)."),
    ("➕ Grupos de venda de kits (entrar com conta)", 5, 5, 2, 1,
     "LEGALIDADE: conta falsa (CP art. 307) e interação com vendedor. ENGANO: fingir ser comprador."),
]
CORES = {1: "🟩", 2: "🟨", 3: "🟧", 4: "🟥", 5: "⬛"}
print(f"{'Fonte':<55} Jur OPS Cus Val")
for nome, j, o, c, v, por in FONTES:
    print(f"{nome:<55} {CORES[j]}{j} {CORES[o]}{o} {CORES[c]}{c} {CORES[6 - v]}{v}")
    print(f"{'':<55} ↳ {por}")
print("\n(Na coluna 'Valor', a cor é invertida: verde = muito valor para um iniciante.)")

tabela = ["| Fonte | Risco jurídico | Risco de OPSEC | Custo | Valor p/ iniciante | Justificativa |", "|---|---|---|---|---|---|"]
tabela += [f"| {n} | {j} | {o} | {c} | {v} | {p} |" for n, j, o, c, v, p in FONTES]
with open("mapa-risco-fontes.md", "w", encoding="utf-8") as f:
    f.write("# Mapa de calor de risco de fontes\n\n" + "\n".join(tabela) + "\n")

try:
    from google.colab import files
    for nome in ["plano-de-coleta-isca.md", "mapa-risco-fontes.md"]:
        files.download(nome)
except (ImportError, FileNotFoundError):
    print("(Fora do Colab: os arquivos ficaram na pasta atual.)")

## ✅ Antes de fechar o Dia 41

- [ ] Registro de instalação: ambiente, `gpgv` completo com `Good signature from`, Security Level, dois onions verificados **pelo seu próprio Tor Browser** e a segunda fonte de cada um
- [ ] Plano de coleta (saúde, do original) e plano da Operação Isca, com 3+ proibições e 2+ paradas (uma é CSAM, com SaferNet Brasil / Disque 100)
- [ ] Mapa de calor com justificativa de legalidade **e** de engano para fóruns fechados
- [ ] Print do anúncio registrado com hash; link de convite **não** seguido
- [ ] Nenhum ambiente criminoso visitado

**Amanhã (Dia 42):** medir a atividade de sites de vazamento de ransomware **por agregadores**, sem coletar vítimas, e responder ao conselho da Serra Azul se "cooperativas são o alvo nº 1".

🧠 Se algo que você viu hoje (ou em qualquer dia) perturbou você, fale com alguém da sua equipe. Isso faz parte do protocolo.

---
<sub>Baseado em cyber-investigator-skills por ThiefDroppers (MIT) · Investigador Cyber 90 Dias por Deivison Lourenço</sub>